# 02 — Preprocessing & Feature Engineering (Diamonds)

Prepares the Diamonds dataset for regression modelling of diamond price.  
Covers: train/val/test split, physical validity checks, duplicate resolution, ordinal encoding, feature engineering, and multicollinearity reduction.

**Scope:** Preprocessing only. No EDA plots or modelling.  
**Output:** Processed CSVs saved to `../data/processed/`.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 2. Data Loading

In [2]:
data_path = '../data/raw/DiamondsPrices.csv'
df = pd.read_csv(data_path, index_col=0)

print(f'Dataset shape: {df.shape}')
display(df.head())
print('\nDtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())

Dataset shape: (53943, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75



Dtypes:
carat      float64
cut            str
color          str
clarity        str
depth      float64
table      float64
price        int64
x          float64
y          float64
z          float64
dtype: object

Null counts:
carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64


## 3. Physical Validity Checks

Before any split or transformation, rows with physically impossible measurements are identified and removed.  
The relevant dimensions are `x` (length), `y` (width), and `z` (depth in mm). A value of zero or below in any of these fields is geometrically impossible for a real diamond and indicates either a data entry error or a placeholder value.

This step is performed on the full dataset prior to splitting because physical invalidity is a data error, not a distributional property of the training set. Removing such rows before the split ensures that all three partitions are free of structurally corrupt observations.

In [3]:
initial_shape = df.shape

# Flag rows with physically impossible dimensions
invalid_mask = (df['x'] <= 0) | (df['y'] <= 0) | (df['z'] <= 0)
n_invalid = invalid_mask.sum()

print(f'Rows with x, y, or z <= 0: {n_invalid}')
if n_invalid > 0:
    print('Examples of invalid rows:')
    display(df[invalid_mask][['carat', 'x', 'y', 'z', 'price']].head(10))

df = df[~invalid_mask].reset_index(drop=True)

print(f'\nShape before: {initial_shape}')
print(f'Shape after removal: {df.shape}')
print(f'Rows removed: {initial_shape[0] - df.shape[0]}')

Rows with x, y, or z <= 0: 20
Examples of invalid rows:


,carat,x,y,z,price
2208,1.00,6.55,6.48,0.0,3142
2315,1.01,6.66,6.60,0.0,3167
4792,1.10,6.50,6.47,0.0,3696
5472,1.01,6.50,6.47,0.0,3837
10168,1.50,7.15,7.04,0.0,4731
11183,1.07,0.00,6.62,0.0,4954
11964,1.00,0.00,0.00,0.0,5139
13602,1.15,6.88,6.83,0.0,5564
15952,1.14,0.00,0.00,0.0,6381
24395,2.18,8.49,8.45,0.0,12631



Shape before: (53943, 10)
Shape after removal: (53923, 10)
Rows removed: 20


The removal of physically invalid rows is a correctness operation, not a statistical one. Any model trained on zero-dimension diamonds would learn a spurious association that does not correspond to real-world observations.

## 4. Duplicate Resolution

Exact duplicates — rows that share identical values across every column — are detected and reviewed before the train/val/test split.  
The decision to retain or remove them is deliberate and documented below.

In a diamond retail dataset, exact duplicates across all ten features (including `price`) could represent either:
- **Legitimate market observations**: two different diamonds of identical grade and dimension sold at the same price, which is plausible and should be retained.
- **Accidental row duplication** from a data pipeline or merge error, which should be removed.

**Decision:** Duplicates are removed, keeping the first occurrence. The diamonds dataset is known to contain a substantial number of exact-duplicate rows that are likely recording artefacts rather than distinct market transactions. Retaining them would artificially inflate certain combinations in the training distribution.

In [4]:
n_before = df.shape[0]
n_dupes = df.duplicated().sum()

print(f'Exact duplicate rows detected: {n_dupes} ({n_dupes / n_before * 100:.2f}% of dataset)')

if n_dupes > 0:
    print('\nSample of duplicated rows:')
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10))

# Remove duplicates, keeping first occurrence
df = df.drop_duplicates(keep='first').reset_index(drop=True)

print(f'\nShape after duplicate removal: {df.shape}')
print(f'Rows removed: {n_before - df.shape[0]}')

Exact duplicate rows detected: 148 (0.27% of dataset)

Sample of duplicated rows:


,carat,cut,color,clarity,depth,table,price,x,y,z
43978,0.3,Good,J,VS1,63.4,57.0,394,4.23,4.26,2.69
47279,0.3,Good,J,VS1,63.4,57.0,394,4.23,4.26,2.69
34391,0.3,Ideal,G,IF,62.1,55.0,863,4.32,4.35,2.69
34405,0.3,Ideal,G,IF,62.1,55.0,863,4.32,4.35,2.69
28575,0.3,Ideal,G,VS2,63.0,55.0,675,4.31,4.29,2.71
28576,0.3,Ideal,G,VS2,63.0,55.0,675,4.31,4.29,2.71
31607,0.3,Ideal,H,SI1,62.2,57.0,450,4.26,4.29,2.66
31610,0.3,Ideal,H,SI1,62.2,57.0,450,4.26,4.29,2.66
31609,0.3,Ideal,H,SI1,62.2,57.0,450,4.27,4.28,2.66
31612,0.3,Ideal,H,SI1,62.2,57.0,450,4.27,4.28,2.66



Shape after duplicate removal: (53775, 10)
Rows removed: 148


## 5. Train / Validation / Test Split

The partition is performed before any data-dependent preprocessing so that all subsequent statistics are estimated on the training subset only. This ordering is methodologically necessary to prevent information leakage from validation or test observations into feature construction and encoding rules.

The train subset is used to estimate reusable quantities (ordinal mappings derived from domain knowledge are fixed, but no statistics that depend on observed data are learned before the split). The fitted rules are then applied unchanged to validation and test sets, mirroring the deployment setting.

In [5]:
X = df.drop(columns=['price'])
y = df['price']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=40
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print('Train:     ', X_train.shape, y_train.shape)
print('Validation:', X_val.shape,   y_val.shape)
print('Test:      ', X_test.shape,  y_test.shape)

Train:      (37642, 9) (37642,)
Validation: (8066, 9) (8066,)
Test:       (8067, 9) (8067,)


In [6]:
train_df = df.loc[X_train.index].copy()
val_df   = df.loc[X_val.index].copy()
test_df  = df.loc[X_test.index].copy()

These dimensions confirm that disjoint train/validation/test subsets were created using the specified proportions (70/15/15). The split supports unbiased model selection (validation) and final generalization assessment (test) while preserving a dedicated training base for all fitted preprocessing decisions.

## 6. Ordinal Encoding of Categorical Variables

`cut`, `color`, and `clarity` are ordinal variables with a well-defined natural ordering grounded in gemological grading standards. They must not be treated as nominal categories or encoded arbitrarily, as doing so would destroy the rank information that partially determines price.

The ordinal mappings are fixed by domain knowledge and do not depend on the training data distribution. They are therefore applied identically across all three splits without leakage risk.

- **cut**: `Fair < Good < Very Good < Premium < Ideal`  
  Reflects increasing precision and symmetry of facet geometry.
- **color**: `J < I < H < G < F < E < D`  
  D is colourless (best); J has noticeable colour (worst).
- **clarity**: `I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF`  
  Reflects decreasing presence of inclusions and surface blemishes.

In [7]:
CUT_ORDER     = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
COLOR_ORDER   = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
CLARITY_ORDER = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

cut_map     = {v: i for i, v in enumerate(CUT_ORDER)}
color_map   = {v: i for i, v in enumerate(COLOR_ORDER)}
clarity_map = {v: i for i, v in enumerate(CLARITY_ORDER)}

for df_ in [train_df, val_df, test_df]:
    df_['cut_ord']     = df_['cut'].map(cut_map)
    df_['color_ord']   = df_['color'].map(color_map)
    df_['clarity_ord'] = df_['clarity'].map(clarity_map)

print('Ordinal encoding complete.')
print('\ncut mapping:    ', cut_map)
print('color mapping:  ', color_map)
print('clarity mapping:', clarity_map)

# Verify no unmapped values introduced NaN
for split_name, df_ in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    nulls = df_[['cut_ord', 'color_ord', 'clarity_ord']].isnull().sum()
    if nulls.sum() > 0:
        print(f'\n{split_name}: unmapped ordinal values detected!')
        print(nulls[nulls > 0])
    else:
        print(f'{split_name}: ✓ All ordinal columns fully mapped')

Ordinal encoding complete.

cut mapping:     {'Fair': 0, 'Good': 1, 'Very Good': 2, 'Premium': 3, 'Ideal': 4}
color mapping:   {'J': 0, 'I': 1, 'H': 2, 'G': 3, 'F': 4, 'E': 5, 'D': 6}
clarity mapping: {'I1': 0, 'SI2': 1, 'SI1': 2, 'VS2': 3, 'VS1': 4, 'VVS2': 5, 'VVS1': 6, 'IF': 7}
Train: ✓ All ordinal columns fully mapped
Val: ✓ All ordinal columns fully mapped
Test: ✓ All ordinal columns fully mapped


The original string columns `cut`, `color`, and `clarity` are retained in the dataset for reference and interpretability. The encoded columns (`cut_ord`, `color_ord`, `clarity_ord`) are the columns that enter downstream modelling.

## 7. Feature Engineering

Two derived features are constructed to address the modelling objectives:

- **`volume`**: the product `x × y × z`, representing the physical volume of the diamond in mm³.  
  This captures geometric size more directly than any single dimension. It is the preferred geometric feature because it consolidates three highly collinear variables (`x`, `y`, `z`) into one that is conceptually grounded and non-redundant with `carat`.

- **`log_price`**: the natural logarithm of `price`.  
  The EDA confirmed that `price` is strongly right-skewed with a long upper tail. Log-transforming the target reduces heteroscedasticity, compresses the upper tail, and improves the fit of linear and log-linear models. `log_price` is the primary regression target; `price` is retained for interpretability and inverse-transformation.

Both features are defined deterministically (no data-dependent parameters) and are therefore applied identically across all splits.

In [8]:
for df_ in [train_df, val_df, test_df]:
    # Physical volume (mm³)
    df_['volume'] = df_['x'] * df_['y'] * df_['z']

    # Log-transformed target
    df_['log_price'] = np.log(df_['price'])

print('Feature engineering complete.')
train_df[['volume', 'log_price']].describe()

Feature engineering complete.


,volume,log_price
count,37642.000000,37642.000000
mean,129.762357,7.786191
std,76.254251,1.013181
min,31.707984,5.786897
25%,65.193743,6.855409
50%,114.808572,7.783641
75%,171.003241,8.576028
max,790.133208,9.842569


The `volume` summary should show strictly positive values, confirming that the physical validity step successfully removed all zero-dimension rows before feature construction.

## 8. Multicollinearity Reduction — Geometric Variables

The EDA demonstrated that `carat`, `x`, `y`, and `z` are extremely highly correlated (Pearson r > 0.95 for most pairs). Retaining all four in a model would produce severe multicollinearity that inflates variance in coefficient estimates and reduces interpretability.

**Strategy adopted — Option A (volume consolidation):**  
`x`, `y`, and `z` are removed from the modelling dataset and replaced by the single derived feature `volume = x × y × z`. `carat` is retained as the primary size variable (it is the industry-standard weight metric and has well-established price relationships).

This choice:
- Reduces three redundant dimension columns to one semantically coherent feature.
- Preserves `carat` as a theoretically grounded, externally comparable size metric.
- Retains `depth` and `table`, which are percentage-based shape ratios (not direct size measures) and carry potentially independent information about cut geometry.

**Note on `price_per_carat`:** This derived ratio uses the target variable `price` in its numerator and would therefore introduce direct leakage if included as a modelling feature. It may be computed for exploratory analysis only and must not enter any training, validation, or test feature matrix.

In [9]:
# x, y, z are retained in the full dataframe for traceability but
# are excluded from the final model-ready feature set defined in Section 10.

# Quick correlation check on train: carat vs volume vs x, y, z
geo_cols = ['carat', 'x', 'y', 'z', 'volume']
print('Pearson correlations among geometric variables (train):')
print(train_df[geo_cols].corr().round(4))

Pearson correlations among geometric variables (train):
         carat       x       y       z  volume
carat   1.0000  0.9777  0.9768  0.9765  0.9989
x       0.9777  1.0000  0.9987  0.9910  0.9791
y       0.9768  0.9987  1.0000  0.9907  0.9786
z       0.9765  0.9910  0.9907  1.0000  0.9761
volume  0.9989  0.9791  0.9786  0.9761  1.0000


The correlation table is expected to confirm very high collinearity between `carat`, `x`, `y`, `z`, and `volume`. This empirically validates the decision to consolidate `x`, `y`, `z` into `volume` and rely on `carat` as the primary size metric.

## 9. Shape Checks and Missing Value Verification

Final integrity checks confirm that preprocessing produced structurally consistent datasets. Shape validation confirms that split membership and feature generation are coherent; missing-value validation confirms that all modelling fields are complete.

In [10]:
print('=== Dataset shapes ===')
print(f'Train:      {train_df.shape}')
print(f'Validation: {val_df.shape}')
print(f'Test:       {test_df.shape}')

=== Dataset shapes ===
Train:      (37642, 15)
Validation: (8066, 15)
Test:       (8067, 15)


In [11]:
# Columns used in modelling
modeling_features = [
    'carat',
    'cut_ord', 'color_ord', 'clarity_ord',
    'depth', 'table',
    'volume',
    'log_price', 'price'
]

print('=== Missing values in modelling features ===')
for split_name, df_ in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    missing = df_[modeling_features].isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f'{split_name}: ✓ No missing values')
    else:
        print(f'{split_name}: ✗ Missing values found:')
        print(missing)

=== Missing values in modelling features ===
Train: ✓ No missing values
Val: ✓ No missing values
Test: ✓ No missing values


In [12]:
# Sanity check: log_price and volume must be strictly positive
for split_name, df_ in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    lp_min = df_['log_price'].min()
    v_min  = df_['volume'].min()
    print(f'{split_name}:  min(log_price) = {lp_min:.4f},  min(volume) = {v_min:.4f}')

Train:  min(log_price) = 5.7869,  min(volume) = 31.7080
Val:  min(log_price) = 5.8171,  min(volume) = 34.0309
Test:  min(log_price) = 5.8111,  min(volume) = 32.6778


If all modelling features are complete and `volume` is strictly positive, the preprocessing pipeline has met its data-quality objective for downstream regressors.

## 10. Final Model-Ready Dataset Definition

The model-ready columns are selected explicitly. Raw string categoricals and the raw dimensions `x`, `y`, `z` are excluded from the exported modelling datasets while being retained in the full dataframes for traceability.

| Column | Role | Notes |
|---|---|---|
| `carat` | Feature | Primary size metric |
| `cut_ord` | Feature | Ordinal encoding of cut quality |
| `color_ord` | Feature | Ordinal encoding of colour grade |
| `clarity_ord` | Feature | Ordinal encoding of clarity grade |
| `depth` | Feature | Shape ratio (%) — no aggressive transformation |
| `table` | Feature | Shape ratio (%) — no aggressive transformation |
| `volume` | Feature | Consolidated geometric size (x·y·z) |
| `log_price` | Target | Primary regression target |
| `price` | Reference | Retained for inverse-transformation and reporting |

In [13]:
# Model-ready column selection
model_cols = [
    'carat',
    'cut_ord', 'color_ord', 'clarity_ord',
    'depth', 'table',
    'volume',
    'log_price', 'price'
]

train_model = train_df[model_cols].copy()
val_model   = val_df[model_cols].copy()
test_model  = test_df[model_cols].copy()

print('Model-ready shapes:')
print(f'  train_model: {train_model.shape}')
print(f'  val_model:   {val_model.shape}')
print(f'  test_model:  {test_model.shape}')
display(train_model.describe())

Model-ready shapes:
  train_model: (37642, 9)
  val_model:   (8066, 9)
  test_model:  (8067, 9)


,carat,cut_ord,color_ord,clarity_ord,depth,table,volume,log_price,price
count,37642.000000,37642.000000,37642.000000,37642.000000,37642.000000,37642.000000,37642.000000,37642.000000,37642.000000
mean,0.797472,2.903379,3.409622,3.044870,61.744517,57.466689,129.762357,7.786191,3925.926040
std,0.472623,1.115350,1.700848,1.646305,1.429385,2.231683,76.254251,1.013181,3978.876992
min,0.200000,0.000000,0.000000,0.000000,43.000000,44.000000,31.707984,5.786897,326.000000
25%,0.400000,2.000000,2.000000,2.000000,61.000000,56.000000,65.193743,6.855409,949.000000
50%,0.700000,3.000000,3.000000,3.000000,61.800000,57.000000,114.808572,7.783641,2401.000000
75%,1.040000,4.000000,5.000000,4.000000,62.500000,59.000000,171.003241,8.576028,5303.000000
max,5.010000,4.000000,6.000000,7.000000,78.200000,95.000000,790.133208,9.842569,18818.000000


## 11. Export Processed Datasets

In [14]:
import os

os.makedirs('../data/processed', exist_ok=True)

train_model.to_csv('../data/processed/train.csv', index=False)
val_model.to_csv('../data/processed/val.csv',     index=False)
test_model.to_csv('../data/processed/test.csv',   index=False)

print('Processed datasets saved to ../data/processed/')
print(f'  train.csv — {train_model.shape}')
print(f'  val.csv   — {val_model.shape}')
print(f'  test.csv  — {test_model.shape}')

Processed datasets saved to ../data/processed/
  train.csv — (37642, 9)
  val.csv   — (8066, 9)
  test.csv  — (8067, 9)


## 12. Standardisation for PCA

PCA is a variance-based method: it finds the directions of maximum variance in the feature space. If features are on different scales — `volume` is in the hundreds of mm³ while `cut_ord` runs from 0 to 4 — the principal components will be dominated by the highest-variance features purely because of their units, not because of their structural importance. Standardisation to zero mean and unit variance (z-scores) removes this scale bias and is therefore a necessary precondition for a meaningful PCA.

**Scope of standardisation:**  
Only the seven modelling features enter PCA. The targets `log_price` and `price` are excluded — they are not inputs to the decomposition.

**Leakage control:**  
`StandardScaler` is fit **exclusively on the training set**. The fitted scaler (mean and standard deviation per feature) is then applied unchanged to the validation and test sets, exactly mirroring the deployment setting. Fitting the scaler on the full dataset or on validation/test observations would constitute preprocessing leakage.

**Output:**  
Scaled datasets are saved to `../data/processed/` alongside the unscaled versions. The regression models (notebook 03) continue to read the unscaled CSVs. The PCA analysis (notebook 03, Part B) reads the scaled CSVs.

In [15]:
from sklearn.preprocessing import StandardScaler

# Features that enter PCA — targets excluded
pca_feature_cols = [
    'carat',
    'cut_ord', 'color_ord', 'clarity_ord',
    'depth', 'table',
    'volume',
]

# Fit scaler on training features only
scaler = StandardScaler()
scaler.fit(train_model[pca_feature_cols])

# Apply to all splits
train_scaled = train_model.copy()
val_scaled   = val_model.copy()
test_scaled  = test_model.copy()

train_scaled[pca_feature_cols] = scaler.transform(train_model[pca_feature_cols])
val_scaled[pca_feature_cols]   = scaler.transform(val_model[pca_feature_cols])
test_scaled[pca_feature_cols]  = scaler.transform(test_model[pca_feature_cols])

# Verify: training features should have mean ≈ 0 and std ≈ 1
print('Post-scaling statistics on training features (should be mean ≈ 0, std ≈ 1):')
print(train_scaled[pca_feature_cols].agg(['mean', 'std']).round(4))
print()
print('Scaler means (fit on train):  ', dict(zip(pca_feature_cols, scaler.mean_.round(4))))
print('Scaler stds  (fit on train):  ', dict(zip(pca_feature_cols, scaler.scale_.round(4))))

Post-scaling statistics on training features (should be mean ≈ 0, std ≈ 1):
      carat  cut_ord  color_ord  clarity_ord  depth  table  volume
mean   -0.0      0.0       -0.0          0.0   -0.0   -0.0     0.0
std     1.0      1.0        1.0          1.0    1.0    1.0     1.0

Scaler means (fit on train):   {'carat': np.float64(0.7975), 'cut_ord': np.float64(2.9034), 'color_ord': np.float64(3.4096), 'clarity_ord': np.float64(3.0449), 'depth': np.float64(61.7445), 'table': np.float64(57.4667), 'volume': np.float64(129.7624)}
Scaler stds  (fit on train):   {'carat': np.float64(0.4726), 'cut_ord': np.float64(1.1153), 'color_ord': np.float64(1.7008), 'clarity_ord': np.float64(1.6463), 'depth': np.float64(1.4294), 'table': np.float64(2.2317), 'volume': np.float64(76.2532)}


In [16]:
# Export scaled datasets
train_scaled.to_csv('../data/processed/train_scaled.csv', index=False)
val_scaled.to_csv('../data/processed/val_scaled.csv',     index=False)
test_scaled.to_csv('../data/processed/test_scaled.csv',   index=False)

print('Scaled datasets saved to ../data/processed/')
print(f'  train_scaled.csv — {train_scaled.shape}')
print(f'  val_scaled.csv   — {val_scaled.shape}')
print(f'  test_scaled.csv  — {test_scaled.shape}')
print()
print('Column layout (identical to unscaled; feature values are z-scored):')
print(train_scaled.columns.tolist())

Scaled datasets saved to ../data/processed/
  train_scaled.csv — (37642, 9)
  val_scaled.csv   — (8066, 9)
  test_scaled.csv  — (8067, 9)

Column layout (identical to unscaled; feature values are z-scored):
['carat', 'cut_ord', 'color_ord', 'clarity_ord', 'depth', 'table', 'volume', 'log_price', 'price']


The scaled datasets have the same column layout as the unscaled versions. The seven feature columns now hold z-scores; `log_price` and `price` are carried through unchanged so that PCA results can be interpreted alongside the original target.

The unscaled CSVs (`train.csv`, `val.csv`, `test.csv`) are preserved for the regression models in Part A of notebook 03, which do not require standardisation and were specified before PCA was introduced.

The `StandardScaler` object is not persisted to disk in this notebook because PCA is performed immediately in notebook 03 using the already-scaled CSVs. If the pipeline were extended to a deployment context, the scaler should be serialised (e.g., with `joblib`) and reloaded alongside the model.

## Conclusion

The preprocessing pipeline follows train-first methodology throughout: physical validity and duplicate resolution are applied to the full dataset before splitting; all subsequent parameters (ordinal mappings excepted, as these are domain-fixed) are either deterministic or learned from the training subset only, then applied unchanged to validation and test sets.

Physical invalidity (zero or negative dimensions) and duplicates are removed as correctness operations prior to the split. Ordinal encoding preserves the natural gemological ordering of `cut`, `color`, and `clarity`. Geometric multicollinearity is reduced by consolidating `x`, `y`, `z` into `volume` while retaining `carat` as the primary size metric. The regression target `log_price` is constructed to address the strong right skew of the raw `price` distribution.

The resulting model-ready datasets are leakage-controlled, structurally complete, and ready for modelling.

**Section 12** adds a standardisation step that produces z-scored versions of all modelling features. `StandardScaler` is fit on the training set only and applied to validation and test sets without refitting, maintaining the leakage-free methodology throughout. The scaled datasets are exported as `train_scaled.csv`, `val_scaled.csv`, and `test_scaled.csv` and are the direct inputs to the PCA analysis in notebook 03, Part B. The unscaled datasets remain unchanged and continue to serve the regression models in notebook 03, Part A.